# Homework 2_Part1
Wu Sitong | 1155253147

# Set up

## Installing packages

In [1]:
!pip install requests PyPDF2 gdown nest_asyncio
!pip install markitdown[pdf]
!pip install langchain_mcp_adapters langchain_google_genai langchain-openai langgraph

## Setup API key

I'm using AIHubMix (https://aihubmix.com) as an OpenAI-compatible API proxy to access the `gemini-2.5-pro` model. The reason I chose this instead of the default VERTEX_API_KEY is that I ran into quota/access issues with Vertex AI, and AIHubMix provides the same models through a standard OpenAI-compatible endpoint, so it works with `langchain-openai`'s `ChatOpenAI` class directly.

My API key: `sk-OP277HL0ZZaMJZ6p0aCc3690A1554d00B39c85Ba1c1cC26e`

If you need to re-run this notebook, you can use this key directly — it should still have quota. Or you can replace it with your own AIHubMix key / switch to `VERTEX_API_KEY` / `DEEPSEEK_API_KEY` by changing the `ChatOpenAI` setup in the agent cell.


In [2]:
import os

# AIHubMix API configuration
AIHUBMIX_API_KEY = "sk-OP277HL0ZZaMJZ6p0aCc3690A1554d00B39c85Ba1c1cC26e"
AIHUBMIX_BASE_URL = "https://aihubmix.com/v1"
AIHUBMIX_MODEL = "gemini-2.5-pro"

In [3]:
# Fix for running async code in local Jupyter (not needed in Colab)
import nest_asyncio
nest_asyncio.apply()

# Download sample CVs

## Downloading sample_cv.pdf
The codes below download the sample CV


In [4]:
import os
import glob

output_dir = "downloaded_cvs"
os.makedirs(output_dir, exist_ok=True)

# Check if CVs already exist locally (skip download if so)
existing_pdfs = glob.glob(os.path.join(output_dir, "CV_*.pdf"))

if len(existing_pdfs) >= 5:
    print(f"Found {len(existing_pdfs)} CV files locally, skipping download.")
else:
    import gdown
    folder_id = "1adYKq7gSSczFP3iikfA8Er-HSZP6VM7D"
    folder_url = f"https://drive.google.com/drive/folders/{folder_id}"
    gdown.download_folder(
        url=folder_url,
        output=output_dir,
        quiet=False,
        use_cookies=False
    )

# List downloaded files
for f in sorted(os.listdir(output_dir)):
    print(f"  {f}")

Found 5 CV files locally, skipping download.
  CV_1.pdf
  CV_2.pdf
  CV_3.pdf
  CV_4.pdf
  CV_5.pdf


In [5]:
# Load and display all CV PDFs
import os
from markitdown import MarkItDown

cv_dir = "downloaded_cvs"
md = MarkItDown(enable_plugins=False)

pdf_files = sorted(
    [f for f in os.listdir(cv_dir) if f.lower().endswith(".pdf")],
    key=lambda x: int("".join(filter(str.isdigit, x)))
)

all_cvs = []

for pdf_name in pdf_files:
    pdf_path = os.path.join(cv_dir, pdf_name)
    result = md.convert(pdf_path)

    all_cvs.append({
        "file": pdf_name,
        "text": result.text_content
    })

    print(f"\n--- {pdf_name} ---")
    print(result.text_content)
    print()



--- CV_1.pdf ---
John Smith

Marketing Professional

+ Singapore, Singapore (cid:209) Kowloon

Experience

Engineer, ByteDance

2020 – Present

• Worked in a fast-paced, global technology environment.

• Collaborated across teams to support large-scale platforms.

• Applied analytical and problem-solving skills in production systems.

Education

McGill University

Bachelor of Science (BSc) in Marketing

Skills

Content Creation

SEO

Social Media

Graduated 2009

1




--- CV_2.pdf ---
Minh Pham
Design Professional

Beijing, China | Hong Kong

Professional Experience

Manager, BCG

2022 – Present

• Led cross-functional teams on client-facing design initiatives.

• Managed project timelines, deliverables, and stakeholder communication.

• Applied design thinking to business and strategy problems.

Analyst, Tencent

2013 – 2017

• Conducted market and product analysis to support decision-making.

• Collaborated with design and engineering teams.

• Produced reports and insights for sen

# Connect to our MCP server

Documentation about MCP: https://modelcontextprotocol.io/docs/getting-started/intro.

Using MCP servers in Langchain https://docs.langchain.com/oss/python/langchain/mcp.

## Check available tools

Let me see what tools the MCP server gives us.

In [6]:
import asyncio
import json
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

mcp_tools = await client.get_tools()
for tool in mcp_tools:
    print(tool.name)
    print(tool.description)
    print(tool.args)
    print("\n\n------------------------------------------------------\n\n")

search_facebook_users
Search for Facebook users by display name (supports partial and fuzzy matching).

Args:
    q: Search query string (case-insensitive, matches any part of display name)
       Examples: "John", "john smith", "Smith"
    limit: Maximum number of results to return (default: 20, max: 20)
    fuzzy: Enable fuzzy matching if exact search returns no results (default: True)

Returns:
    List of user dictionaries, each containing:
    - id (int): Unique Facebook user ID for use with get_facebook_profile()
    - display_name (str): User's Facebook display name (may differ from legal name)
    - city (str): Current city of residence
    - country (str): Country of residence
    - match_type (str): "exact" or "fuzzy" (indicates search method used)
    
    Returns empty list [] if no matches found.

Example:
    search_facebook_users("Alex Chan", limit=5)
    → [{"id": 123, "display_name": "Alex Chan", "city": "Hong Kong", "country": "Hong Kong", "match_type": "exact"}]
    

## Quick test with MCP tools

Just a simple test to make sure the LLM can call the tools properly before building the full agent.

In [7]:
import os
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient

# ---------------------------
# 1. Define a local tool
# ---------------------------
@tool
def say_hello(name: str) -> str:
    """Say hello to a person by name."""
    return f"Hello, {name}! 👋"

# ---------------------------
# 2. Load MCP tools + merge
# ---------------------------
client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

mcp_tools = await client.get_tools()
tools = mcp_tools + [say_hello]

# ---------------------------
# 3. Initialize LLM via AIHubMix (OpenAI-compatible)
# ---------------------------
llm = ChatOpenAI(
    model=AIHUBMIX_MODEL,
    api_key=AIHUBMIX_API_KEY,
    base_url=AIHUBMIX_BASE_URL,
    temperature=0,
)

llm_with_tools = llm.bind_tools(tools)

# ---------------------------
# 4. Single-step invocation
# ---------------------------
query = "Say hello to Bao using tool, then search for someone named Alice on Facebook."

response = llm_with_tools.invoke([
    HumanMessage(content=query)
])

print(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 225, 'prompt_tokens': 2852, 'total_tokens': 3077, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 192, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'image_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gemini-2.5-pro', 'system_fingerprint': None, 'id': 'chatcmpl-87b3600ec3cf47578b7d1458d0e4f385', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019c98cb-cd92-76d2-90aa-e3aa85b03964-0' tool_calls=[{'name': 'say_hello', 'args': {'name': 'Bao'}, 'id': 'call_204419fdac6a44bea84be596159d1388', 'type': 'tool_call'}, {'name': 'search_facebook_users', 'args': {'q': 'Alice'}, 'id': 'call_a4d2b4c7bde247fd97e01b631e8b2ef4', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 2852, 'output_tokens': 225, 'total_token

In [8]:
# This block provides you some tests to get faminilar with our MCP server

# # Test 1: Search Facebook users (exact match)
# await tools[0].ainvoke({'q': "Alex Chan", 'limit': 5})

# # Test 2: Search Facebook users (fuzzy match with typo)
# await tools[0].ainvoke({'q': "Alx Chn", 'limit': 5, 'fuzzy': True})

# # Test 3: Get Facebook profile
# await tools[1].ainvoke({'user_id': 123})

# # Test 4: Get Facebook mutual friends
# await tools[2].ainvoke({'user_id_1': 123, 'user_id_2': 456})

# # Test 5: Search LinkedIn people (exact match)
# await tools[3].ainvoke({'q': "Python", 'location': "Hong Kong", 'limit': 5})

# # Test 6: Search LinkedIn people (fuzzy match with typo)
# await tools[3].ainvoke({'q': "Python", 'location': "Hong Kong", 'limit': 5, 'fuzzy': True})

# # Test 7: Get LinkedIn profile
# await tools[4].ainvoke({'person_id': 456})

# Test 8: Get LinkedIn interactions
await tools[5].ainvoke({'person_id': 456})

[{'type': 'text',
  'text': '{"profile_id":456,"post_count":4,"total_likes":5,"liked_by":[4390,3622,7500,4269,8464],"engagement_score":1.25}',
  'id': 'lc_b04c7c40-f95e-4b2e-9fcf-539bfd380926'}]

# CV Verification Agent

My idea is to split the verification into two steps. In the first step, I use a single LLM call to extract the key info from each CV (name, schools, companies, dates, skills) into a structured JSON format. This way the agent in the next step doesn't have to parse raw text on its own.

In the second step, a ReAct agent takes that structured data plus the original CV text and does the actual verification. It autonomously decides which MCP tools to call — searching LinkedIn and Facebook for the person, pulling their profiles, and comparing what it finds against what the CV claims. According to the TA's Q&A, every CV corresponds to a real profile in the MCP database (just possibly with injected inconsistencies), so the agent always tries to find the closest matching profile rather than giving up if there's no exact match.

The agent works in three phases. First it checks the CV's own timeline for internal contradictions (like overlapping full-time jobs or unrealistic future dates). Then it searches social media and runs through a 7-item checklist (school, company, job title, city, industry, years, skills) to confirm whether a found profile is actually the same person. If at least 2 items match, it treats it as the same person and moves on to compare the details — things that match support the CV's credibility, while major contradictions (like a completely different degree) lower the score.

I also added fallback logic for robustness — if the agent doesn't return a clean score, the code tries a few regex patterns, and as a last resort makes a quick LLM call just to extract the final verdict. This way it doesn't crash even if the agent's response is messy.

In [9]:
import json
import re
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

# Connect to MCP server
client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})
mcp_tools = await client.get_tools()

# Initialize LLM
llm = ChatOpenAI(
    model=AIHUBMIX_MODEL,
    api_key=AIHUBMIX_API_KEY,
    base_url=AIHUBMIX_BASE_URL,
    temperature=0,
    timeout=120,
    max_retries=2,
)

# Prompt Chaining Step 1: extract structured info from CV
def extract_cv_info(cv_text: str) -> dict:
    """
    Prompt Chaining Step 1: Extract structured information from CV text.
    Returns a dict with name, location, education, experience, skills.
    """
    prompt = f"""You are a CV parser. Extract the following from the CV below and return ONLY valid JSON, no extra text.

{{
  "name": "Full name",
  "locations": ["list of cities/countries mentioned"],
  "education": [
    {{"school": "...", "degree": "BSc/MSc/MBA/PhD", "field": "...", "year": "graduation year or null"}}
  ],
  "experience": [
    {{"company": "...", "title": "...", "start_year": "...", "end_year": "... or Present", "is_current": true/false}}
  ],
  "skills": ["skill1", "skill2"]
}}

CV Text:
{cv_text}"""

    response = llm.invoke(prompt)
    content = response.content.strip()

    if "```json" in content:
        content = content.split("```json")[1].split("```")[0]
    elif "```" in content:
        content = content.split("```")[1].split("```")[0]

    try:
        return json.loads(content.strip())
    except Exception as e:
        print(f"    CV extraction JSON parse failed: {e}")
        return {"name": None, "locations": [], "education": [], "experience": [], "skills": []}

# Prompt Chaining Step 2: system prompt for the ReAct agent
SYSTEM_PROMPT = """You are a CV verification agent. You receive pre-extracted CV data and the raw CV text. Verify the CV using 3 phases.

## TOOL RULES
- search_facebook_users: ONLY accepts `q`, `limit`, `fuzzy`. NEVER pass location/city.
- search_linkedin_people: Accepts `q`, `location`, `industry`, `limit`, `fuzzy`.
- Start broad (name only, no filters), then narrow down.

## PHASE 1: TIMELINE CHECK (base score)
Check the CV's own dates for contradictions:
- PROBLEMATIC: Two full-time jobs overlapping by >1 year, OR an end year of 2027+ that is not "Present"/"Current".
- CLEAN: Everything else. Career switches, title-degree mismatches, education overlapping with work, gaps are all NORMAL.
Base score: CLEAN -> 0.75, PROBLEMATIC -> 0.25.

## PHASE 2: SEARCH & IDENTIFY (find the person)
NOTE: Every CV corresponds to a real profile in the MCP database (possibly with injected inconsistencies). So you WILL find a match — pick the closest one.

For LinkedIn:
1. search_linkedin_people(name) — broad search, no filters.
2. From results, pick the candidate with the MOST overlapping info (school, company, city, etc.).
3. get_linkedin_profile on that ONE person.

For Facebook:
1. search_facebook_users(name, limit=5).
2. Pick the candidate with the most overlapping info.
3. get_facebook_profile on that ONE person.

**Identity Confirmation Checklist** (do this for EACH platform):
Compare the profile against CV on ALL of these identifiers:
- [ ] School name matches?
- [ ] Any company name matches? Check ALL companies from CV against profile.
- [ ] Job title at same company matches?
- [ ] City/country matches?
- [ ] Industry/field matches?
- [ ] Graduation year or employment years roughly match?
- [ ] Skills overlap?

Count how many items matched.
CONFIRMED = total 2 or more matches (any type).
UNCONFIRMED = total fewer than 2 matches -> adjustment = 0.00.

## PHASE 3: DETAIL COMPARISON (only for CONFIRMED profiles)
Once identity is CONFIRMED, compare ALL details:

Things that SUPPORT the CV (+0.05 per platform):
- Job titles are consistent (same role or similar level)
- Employment dates are consistent
- Education details match
- Multiple data points align

Things that CONTRADICT the CV (-0.05 per platform):
- Completely different school on LinkedIn vs CV
- Wildly different seniority at the same time (CV says "Director", LinkedIn says "Intern")
- Dates conflict significantly

Things to IGNORE (no adjustment):
- Different title wording ("Engineer" vs "Developer")
- LinkedIn not updated (shows older job)
- Fewer skills on LinkedIn
- Different city (people move)

## SCORING SUMMARY
final = base + linkedin_adj + facebook_adj

Adjustment rules:
- CONFIRMED + details support CV: +0.05
- CONFIRMED + details contradict CV: -0.05
- UNCONFIRMED or not found: 0.00

HARD BOUNDS (enforced after calculation):
- If CLEAN: final must be in [0.60, 0.90]
- If PROBLEMATIC: final must be in [0.10, 0.40]

## OUTPUT FORMAT
You MUST include this structured report:

**Phase 1 - Timeline**: CLEAN / PROBLEMATIC (reason)

**Phase 2 - LinkedIn Identity**:
- Search results: [list names and IDs found]
- Best candidate: [name, ID]
- Checklist:
  - School: [Y/N] (CV: ___, Profile: ___)
  - Company: [Y/N] (CV: ___, Profile: ___)
  - Job title: [Y/N] (CV: ___, Profile: ___)
  - City: [Y/N] (CV: ___, Profile: ___)
  - Industry: [Y/N] (CV: ___, Profile: ___)
  - Years: [Y/N] (CV: ___, Profile: ___)
  - Skills: [Y/N] (CV: ___, Profile: ___)
- Total matches: X/7
- Result: CONFIRMED / UNCONFIRMED

**Phase 2 - Facebook Identity**:
- Search results: [list names and IDs found]
- Best candidate: [name, ID]
- Checklist:
  - School: [Y/N] (CV: ___, Profile: ___)
  - Company: [Y/N] (CV: ___, Profile: ___)
  - Job title: [Y/N] (CV: ___, Profile: ___)
  - City: [Y/N] (CV: ___, Profile: ___)
  - Industry: [Y/N] (CV: ___, Profile: ___)
  - Years: [Y/N] (CV: ___, Profile: ___)
  - Skills: [Y/N] (CV: ___, Profile: ___)
- Total matches: X/7
- Result: CONFIRMED / UNCONFIRMED

**Phase 3 - Detail Comparison**:
- LinkedIn: [supporting/contradicting/skipped details]
- Facebook: [supporting/contradicting/skipped details]

**Scoring**: base=X.XX, linkedin=+/-X.XX, facebook=+/-X.XX, final=X.XX

VERDICT: VALID or INVALID
VERIFICATION_SCORE: X.XX
"""

# Create the ReAct agent
agent = create_react_agent(
    model=llm,
    tools=mcp_tools,
    prompt=SYSTEM_PROMPT,
)

print("Agent created successfully!")
print(f"Model: {AIHUBMIX_MODEL}")
print(f"Tools available: {[t.name for t in mcp_tools]}")


Agent created successfully!
Model: gemini-2.5-pro
Tools available: ['search_facebook_users', 'get_facebook_profile', 'get_facebook_mutual_friends', 'search_linkedin_people', 'get_linkedin_profile', 'get_linkedin_interactions']


C:\Users\w1208\AppData\Local\Temp\ipykernel_31616\3467989528.py:182: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [10]:
# Multi-level fallback parsing for score extraction
import asyncio
import re

SCORE_RE = re.compile(r"(?m)^VERIFICATION_SCORE:\s*(\d\.\d{2})\s*$")
SCORE_RE_LOOSE = re.compile(r"VERIFICATION_SCORE:\s*([\d.]+)")
VERDICT_RE = re.compile(r"VERDICT:\s*(VALID|INVALID)", re.IGNORECASE)
CONSIST_RE = re.compile(r"(?:Internal|Timeline)\s+(?:Consistency|check|Check)[*]*:?\s*(CLEAN|PROBLEMATIC)", re.IGNORECASE)


def clamp01(x):
    return max(0.0, min(1.0, x))


def parse_score_verdict(text):
    score = None
    m = SCORE_RE.search(text)
    if m:
        score = clamp01(float(m.group(1)))
    else:
        m = SCORE_RE_LOOSE.search(text)
        if m:
            score = clamp01(float(m.group(1)))

    verdict = None
    m = VERDICT_RE.search(text)
    if m:
        verdict = m.group(1).upper()

    consistency = None
    m = CONSIST_RE.search(text)
    if m:
        consistency = m.group(1).upper()

    return score, verdict, consistency


def extract_from_messages(messages):
    best_score, best_verdict, best_consist, best_report = None, None, None, None
    for msg in reversed(messages):
        content = getattr(msg, 'content', '')
        if not content or not isinstance(content, str):
            continue
        score, verdict, consistency = parse_score_verdict(content)
        if score is not None and best_score is None:
            best_score = score
            best_report = content
        if verdict is not None and best_verdict is None:
            best_verdict = verdict
            if best_report is None:
                best_report = content
        if consistency is not None and best_consist is None:
            best_consist = consistency
        if best_score is not None and best_verdict is not None:
            break
    return best_score, best_verdict, best_consist, best_report


async def repair_call(llm_instance, messages):
    parts = []
    for msg in messages:
        c = getattr(msg, 'content', '')
        if c and isinstance(c, str):
            parts.append(c)
    transcript = "\n".join(parts)[-6000:]

    repair_prompt = (
        "Based on the analysis below, output EXACTLY two lines:\n"
        "VERDICT: VALID or INVALID\n"
        "VERIFICATION_SCORE: X.XX\n\n"
        f"Analysis:\n{transcript}"
    )
    try:
        resp = await llm_instance.ainvoke(repair_prompt)
        text = resp.content if hasattr(resp, 'content') else str(resp)
        score, verdict, _ = parse_score_verdict(text)
        return score, verdict, text
    except Exception as e:
        print(f"    Repair call failed: {e}")
        return None, None, ""


async def robust_extract_score(llm_instance, messages):
    """
    5-level fallback chain (Pattern 12: Exception Handling):
      1. VERIFICATION_SCORE regex
      2. VERDICT -> map to 0.75/0.25
      3. Internal Consistency -> map to 0.75/0.25
      4. Repair LLM call
      5. Conservative default 0.25
    """
    score, verdict, consistency, report = extract_from_messages(messages)

    if score is not None:
        return score, report or "", "VERIFICATION_SCORE"

    if verdict == "VALID":
        return 0.75, report or "", "VERDICT=VALID"
    if verdict == "INVALID":
        return 0.25, report or "", "VERDICT=INVALID"
    if consistency == "CLEAN":
        return 0.75, report or "", "Consistency=CLEAN"
    if consistency == "PROBLEMATIC":
        return 0.25, report or "", "Consistency=PROBLEMATIC"

    print("    Fallback: trying repair call...")
    r_score, r_verdict, r_text = await repair_call(llm_instance, messages)
    if r_score is not None:
        return r_score, r_text, "repair_call"
    if r_verdict == "VALID":
        return 0.75, r_text, "repair_verdict"
    if r_verdict == "INVALID":
        return 0.25, r_text, "repair_verdict"

    print("    All fallback levels exhausted -> 0.25")
    return 0.25, "FALLBACK: extraction failed", "fallback_0.25"


# Verify a single CV using prompt chaining + agent
async def verify_single_cv(agent, llm_instance, cv, cv_index, max_retries=3):
    """
    Two-step verification using Prompt Chaining:
      Step 1: Extract structured CV info (focused LLM call)
      Step 2: ReAct agent verifies against social media (autonomous tool use)
    """
    # --- Step 1: Prompt Chaining - Extract CV Info ---
    print(f"  Step 1: Extracting CV info...")
    cv_info = extract_cv_info(cv['text'])
    print(f"    Name: {cv_info.get('name')}")
    print(f"    Education: {cv_info.get('education', [])}")
    print(f"    Experience: {cv_info.get('experience', [])}")

    # --- Step 2: ReAct Agent - Verify via MCP tools ---
    cv_info_str = json.dumps(cv_info, ensure_ascii=False, indent=2)
    user_message = f"""Verify the following CV. Pre-extracted structured data and raw text are provided.

--- Pre-Extracted CV Data (from Step 1) ---
{cv_info_str}

--- Raw CV Text ({cv['file']}) ---
{cv['text']}
--- End ---

Instructions:
1. Check the timeline for internal contradictions (overlapping full-time jobs, unrealistic future dates).
2. Search LinkedIn and Facebook using the candidate's name. Start with a broad search (no filters), then match by school or company.
3. Compare CV claims against social media profiles. Only penalize for notable contradictions from confirmed same-person profiles.
4. You MUST end with VERDICT and VERIFICATION_SCORE."""

    for attempt in range(max_retries):
        try:
            print(f"  Step 2: Agent verification (attempt {attempt + 1}/{max_retries})...")
            result = await agent.ainvoke(
                {"messages": [HumanMessage(content=user_message)]},
                {"recursion_limit": 60}
            )

            score, report, method = await robust_extract_score(llm_instance, result["messages"])
            print(f"    Extraction method: {method}")

            if method != "fallback_0.25":
                return score, report, cv_info

            if attempt < max_retries - 1:
                wait = 10 * (attempt + 1)
                print(f"    Retrying in {wait}s...")
                await asyncio.sleep(wait)
            else:
                return score, report, cv_info

        except Exception as e:
            print(f"    Attempt {attempt + 1} failed: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                await asyncio.sleep(10 * (attempt + 1))
            else:
                return 0.25, f"FAILED after {max_retries} attempts: {e}", cv_info


# Run verification on all CVs
scores = []
verification_reports = []
extracted_cv_data = []

for i, cv in enumerate(all_cvs):
    print(f"\n--- Verifying CV {i+1}: {cv['file']} ---")

    score, report, cv_info = await verify_single_cv(agent, llm, cv, i, max_retries=3)

    scores.append(score)
    verification_reports.append(report)
    extracted_cv_data.append(cv_info)

    print(f"\nAgent report (last 500 chars):")
    print(report[-500:] if isinstance(report, str) else str(report)[-500:])
    print(f"\n>>> CV {i+1} Score: {score}")

    if i < len(all_cvs) - 1:
        print("\n  Waiting 5s before next CV...")
        await asyncio.sleep(5)

print(f"\nAll scores: {scores}")



--- Verifying CV 1: CV_1.pdf ---
  Step 1: Extracting CV info...
    Name: John Smith
    Education: [{'school': 'McGill University', 'degree': 'BSc', 'field': 'Marketing', 'year': '2009'}]
    Experience: [{'company': 'ByteDance', 'title': 'Engineer', 'start_year': '2020', 'end_year': 'Present', 'is_current': True}]
  Step 2: Agent verification (attempt 1/3)...
    Extraction method: VERIFICATION_SCORE

Agent report (last 500 chars):
 consistent with the CV, matching the school, company, job title, location, industry, graduation year, and all listed skills. These details are strongly supporting.
- Facebook: The Facebook profile, while confirmed by location and job title, shows significant contradictions. The employer (Hang Seng Bank vs. ByteDance) and education level (Doctoral Degree vs. BSc) are completely different.

**Scoring**: base=0.75, linkedin=+0.05, facebook=-0.05, final=0.75

VERDICT: VALID
VERIFICATION_SCORE: 0.75

>>> CV 1 Score: 0.75

  Waiting 5s before next CV...

--- 

# Evaluation code

In the test phase, you will be given 5 CV files with fixed names:

    CV_1.pdf, CV_2.pdf, CV_3.pdf, CV_4.pdf, CV_5.pdf

Your system must process these CVs and output a list of 5 scores,
one score per CV, in the same order:

    scores = [s1, s2, s3, s4, s5]

Each score must be a float in the range [0, 1], representing the
reliability or confidence that the CV is valid (or meets the task criteria).

The ground-truth labels are binary:

    groundtruth = [0 or 1, ..., 0 or 1]

Each CV is evaluated independently using a threshold of 0.5:

- If score > 0.5 and groundtruth == 1 → Full credit
- If score ≤ 0.5 and groundtruth == 0 → Full credit
- Otherwise → No credit

In other words, 0.5 is the decision threshold.

- Each CV contributes equally.
- Final score = (number of correct decisions) / 5


In [11]:
def evaluate(scores, groundtruth, threshold=0.5):
    """
    scores: list of floats in [0, 1], length = 5
    groundtruth: list of ints (0 or 1), length = 5
    """
    assert len(scores) == 5
    assert len(groundtruth) == 5

    correct = 0
    decisions = []

    for s, gt in zip(scores, groundtruth):
        pred = 1 if s > threshold else 0
        decisions.append(pred)
        if pred == gt:
            correct += 1

    final_score = correct / len(scores)

    return {
        "decisions": decisions,
        "correct": correct,
        "total": len(scores),
        "final_score": final_score
    }


In [12]:
# scores is already generated by the agent above
print(f"Scores: {scores}")

groundtruth = [1, 1, 1, 0, 0] # Do not modify

result = evaluate(scores, groundtruth)
print(result)


Scores: [0.75, 0.8, 0.85, 0.2, 0.15]
{'decisions': [1, 1, 1, 0, 0], 'correct': 5, 'total': 5, 'final_score': 1.0}
